In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03 — Features de Contexto Estadual
# MAGIC Enriquece a base municipal com posição relativa no estado

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Contexto estadual de 2023

# COMMAND ----------

uf_2023 = (
    spark.table(f"{CATALOGO}.default.gold_indicador_uf")
         .filter("ano = 2023")
         .select(
             F.col("sigla_uf"),
             F.col("taxa_observada").alias("taxa_uf_2023"),
             F.col("meta_do_ano").alias("meta_uf_2023"),
         )
)
print(f"UFs com contexto 2023: {uf_2023.count()}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Base municipal com UF
# MAGIC O `id_municipio` do IBGE tem 7 dígitos; os 2 primeiros são o código da UF.

# COMMAND ----------

# Mapa código IBGE -> sigla UF
MAPA_UF = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL",
    "28": "SE", "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS", "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}

mapa_expr = F.create_map([F.lit(k) for kv in MAPA_UF.items() for k in kv])

base = (
    spark.table(f"{CATALOGO}.default.fase3_features_municipio")
         .withColumn("sigla_uf", mapa_expr[F.substring(F.col("id_municipio"), 1, 2)])
)

print(f"Base: {base.count()} registros")
base.groupBy("sigla_uf").count().orderBy(F.desc("count")).show(30)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Join com contexto estadual e features relativas

# COMMAND ----------

enriquecida = (
    base.join(uf_2023, on="sigla_uf", how="left")
        .withColumn("gap_uf_2023", F.col("taxa_uf_2023") - F.col("meta_uf_2023"))
        .withColumn("posicao_relativa_uf", F.col("taxa_2023") - F.col("taxa_uf_2023"))
        .withColumn("razao_municipio_uf",
                    F.when(F.col("taxa_uf_2023") > 0,
                           F.col("taxa_2023") / F.col("taxa_uf_2023")))
        .withColumn("esforco_vs_uf",
                    F.col("esforco_necessario") - F.col("gap_uf_2023"))
)

print(f"Base enriquecida: {enriquecida.count()}")
enriquecida.select(
    "sigla_uf", "taxa_2023", "taxa_uf_2023", "posicao_relativa_uf",
    "esforco_necessario", "esforco_vs_uf", "target"
).describe().show()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Verificar nulos no join

# COMMAND ----------

enriquecida.select([
    F.round(F.sum(F.col(c).isNull().cast("int")) / enriquecida.count() * 100, 2).alias(c)
    for c in ["sigla_uf", "taxa_uf_2023", "posicao_relativa_uf", "esforco_vs_uf"]
]).show()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Persistir

# COMMAND ----------

(enriquecida.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOGO}.default.fase3_features_contexto"))

print("Tabela criada: default.fase3_features_contexto")

In [0]:
uf_2023 = (
    spark.table(f"{CATALOGO}.default.gold_indicador_uf")
         .filter("ano = 2023")
         .select(
             F.col("sigla_uf"),
             F.col("taxa_observada").alias("taxa_uf_2023"),
         )
)
print(f"UFs com contexto 2023: {uf_2023.count()}")

In [0]:
MAPA_UF = {
    "11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
    "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL",
    "28": "SE", "29": "BA", "31": "MG", "32": "ES", "33": "RJ", "35": "SP",
    "41": "PR", "42": "SC", "43": "RS", "50": "MS", "51": "MT", "52": "GO", "53": "DF",
}

mapa_expr = F.create_map([F.lit(k) for kv in MAPA_UF.items() for k in kv])

base = (
    spark.table(f"{CATALOGO}.default.fase3_features_municipio")
         .withColumn("sigla_uf", mapa_expr[F.substring(F.col("id_municipio"), 1, 2)])
)

print(f"Base: {base.count()} registros")
base.groupBy("sigla_uf").count().orderBy(F.desc("count")).show(30)

In [0]:
enriquecida = (
    base.join(uf_2023, on="sigla_uf", how="left")
        .withColumn("posicao_relativa_uf", F.col("taxa_2023") - F.col("taxa_uf_2023"))
        .withColumn("razao_municipio_uf",
                    F.when(F.col("taxa_uf_2023") > 0,
                           F.col("taxa_2023") / F.col("taxa_uf_2023")))
        .withColumn("meta_vs_uf", F.col("meta_do_ano") - F.col("taxa_uf_2023"))
        .withColumn("esforco_vs_uf",
                    F.col("esforco_necessario") - (F.col("meta_do_ano") - F.col("taxa_uf_2023")))
)

print(f"Base enriquecida: {enriquecida.count()}")
enriquecida.select(
    "sigla_uf", "taxa_2023", "taxa_uf_2023", "posicao_relativa_uf",
    "meta_vs_uf", "esforco_vs_uf", "target"
).describe().show()

In [0]:
enriquecida.select([
    F.round(F.sum(F.col(c).isNull().cast("int")) / enriquecida.count() * 100, 2).alias(c)
    for c in ["sigla_uf", "taxa_uf_2023", "posicao_relativa_uf", "meta_vs_uf", "esforco_vs_uf"]
]).show()

In [0]:
(enriquecida.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOGO}.default.fase3_features_contexto"))

print("Tabela criada: default.fase3_features_contexto")